# Lagrange Vierlinge - Vier verschiedene Primfaktor-Familien

Dieses Notebook zeigt nur Vierlinge (e, a, b, c) für quadratfreie Zahlen, bei denen die Primfaktoren zu vier verschiedenen Familien (mod 12) gehören.

In [ ]:
# ============================================================================
# EINGABEPARAMETER - Hier können Sie die Parameter einfach anpassen
# ============================================================================
limit = 100000  # Obere Grenze für die zu prüfenden Zahlen
output_filename = "energiedoku_viertupel_gefiltert.csv"  # Optional: CSV-Datei speichern
progress_interval = 50000  # Fortschrittsanzeige alle X Zahlen
save_to_csv = True  # Wenn True, werden Ergebnisse in CSV gespeichert

# ============================================================================
# IMPORTS
# ============================================================================
from sage.all import is_squarefree, Integer
import csv

# ============================================================================
# HILFSFUNKTIONEN - Optimiert für schnelle Prüfung
# ============================================================================
def get_family_mod12_fast(x):
    """
    Optimierte Version: Bestimmt die Familie einer Zahl basierend auf modulo 12.
    Gibt direkt den Rest zurück (1, 5, 7, 11) oder None.
    
    Args:
        x: Zahl (int)
    
    Returns:
        1 für x ≡ 1 (mod 12) -> Familie 'e'
        5 für x ≡ 5 (mod 12) -> Familie 'a'
        7 für x ≡ 7 (mod 12) -> Familie 'b'
        11 für x ≡ 11 (mod 12) -> Familie 'c'
        None für andere Reste
    """
    r = abs(x) % 12
    if r in (1, 5, 7, 11):
        return r
    return None

def all_coordinates_different_families_optimized(e, a, b, c):
    """
    Optimierte Prüfung: Ob alle vier Koordinaten zu unterschiedlichen Familien gehören.
    Bricht früh ab, wenn eine Koordinate nicht zu einer Familie gehört.
    
    Args:
        e, a, b, c: Koordinaten (int)
    
    Returns:
        True, wenn alle vier Koordinaten zu unterschiedlichen Familien gehören,
        False sonst
    """
    # Prüfe jede Koordinate einzeln und sammle Familien
    fam_e = get_family_mod12_fast(e)
    if fam_e is None:
        return False
    
    fam_a = get_family_mod12_fast(a)
    if fam_a is None or fam_a == fam_e:
        return False
    
    fam_b = get_family_mod12_fast(b)
    if fam_b is None or fam_b in (fam_e, fam_a):
        return False
    
    fam_c = get_family_mod12_fast(c)
    if fam_c is None or fam_c in (fam_e, fam_a, fam_b):
        return False
    
    # Alle vier sind unterschiedlich und gehören zu Familien
    return True

def prime_factors_have_four_different_families(n):
    """
    Prüft, ob die Primfaktoren einer Zahl zu vier verschiedenen Familien (mod 12) gehören.
    WICHTIG: ALLE Primfaktoren müssen zu den vier Familien gehören (keine anderen wie 2, 3).
    
    Args:
        n: Natürliche Zahl (int oder Sage Integer)
    
    Returns:
        True, wenn die Primfaktoren zu genau vier verschiedenen Familien gehören
        und ALLE Primfaktoren zu diesen Familien gehören,
        False sonst
    """
    try:
        from sage.all import factor
        
        # Faktorisiere die Zahl
        if not isinstance(n, Integer):
            n = Integer(n)
        
        factors = factor(n)
        
        # Sammle die Familien der Primfaktoren
        families = set()
        for prime, exp in factors:
            prime_val = int(prime)
            fam = get_family_mod12_fast(prime_val)
            if fam is None:
                # Wenn ein Primfaktor nicht zu einer Familie gehört (z.B. 2, 3), ist es ungültig
                return False
            families.add(fam)
        
        # Prüfe, ob es genau vier verschiedene Familien gibt
        return len(families) == 4
        
    except Exception as e:
        return False

def get_family_name(r):
    """Konvertiert Rest (1, 5, 7, 11) zu Familienname ('e', 'a', 'b', 'c')"""
    mapping = {1: 'e', 5: 'a', 7: 'b', 11: 'c'}
    return mapping.get(r, '?')

def four_number_sum(n):
    """
    Zerlegt eine natürliche Zahl n in eine Summe von 4 Quadraten.
    
    Args:
        n: Natürliche Zahl (int oder Sage Integer)
    
    Returns:
        Liste [e, a, b, c] mit e² + a² + b² + c² = n, oder None bei Fehler
    """
    try:
        # Konvertiere zu Sage Integer falls nötig
        if not isinstance(n, Integer):
            n = Integer(n)
        
        # Versuche Sage-Methode (falls verfügbar)
        if hasattr(n, 'is_sum_of_four_squares'):
            res = n.is_sum_of_four_squares(ext=True)
            return list(res)
        
        # Fallback: Klassischer Algorithmus
        n_val = int(n)
        if n_val == 0:
            return [0, 0, 0, 0]
        
        # Finde die größte Quadratzahl <= n
        sqrt_n = int(n_val ** 0.5)
        for e in range(sqrt_n, -1, -1):
            remainder = n_val - e**2
            if remainder == 0:
                return [e, 0, 0, 0]
            
            # Versuche 3 Quadrate für den Rest
            sqrt_r = int(remainder ** 0.5)
            for a in range(sqrt_r, -1, -1):
                remainder2 = remainder - a**2
                if remainder2 == 0:
                    return [e, a, 0, 0]
                
                sqrt_r2 = int(remainder2 ** 0.5)
                for b in range(sqrt_r2, -1, -1):
                    remainder3 = remainder2 - b**2
                    if remainder3 >= 0:
                        c = int(remainder3 ** 0.5)
                        if c**2 == remainder3:
                            return [e, a, b, c]
        
        return None
        
    except Exception as e:
        return None

def four_number_sum_with_family_check(n):
    """
    Zerlegt eine natürliche Zahl n in eine Summe von 4 Quadraten,
    wobei alle vier Koordinaten zu unterschiedlichen Familien (mod 12) gehören müssen.
    
    Diese Funktion probiert zuerst die Standard-Zerlegung mit Permutationen und Vorzeichen,
    und nur bei Bedarf alle möglichen Zerlegungen durch.
    
    Args:
        n: Natürliche Zahl (int oder Sage Integer)
    
    Returns:
        Liste [e, a, b, c] mit e² + a² + b² + c² = n und alle zu unterschiedlichen Familien,
        oder None wenn keine solche Zerlegung existiert
    """
    try:
        n_val = int(n)
        if n_val == 0:
            return None  # 0 kann nicht die Bedingung erfüllen
        
        # Import für Permutationen
        from itertools import permutations, product
        
        # OPTIMIERUNG: Versuche zuerst die Standard-Zerlegung
        first_decomp = four_number_sum(n)
        if first_decomp:
            # Teste alle Permutationen und Vorzeichenkombinationen der ersten Zerlegung
            for signs in product([-1, 1], repeat=4):
                signed_decomp = [signs[i] * first_decomp[i] for i in range(4)]
                for perm in permutations(signed_decomp):
                    if all_coordinates_different_families_optimized(*perm):
                        return list(perm)
        
        # Falls die erste Zerlegung nicht funktioniert, probiere alle möglichen Zerlegungen
        # (nur für kleinere Zahlen, um Performance zu gewährleisten)
        if n_val > 100000:  # Für große Zahlen, probiere nur begrenzt
            return None
        
        sqrt_n = int(n_val ** 0.5)
        decompositions = []
        
        # Sammle alle möglichen Zerlegungen (begrenzt auf vernünftige Größe)
        for e in range(min(sqrt_n + 1, 100)):
            for a in range(min(sqrt_n + 1, 100)):
                for b in range(min(sqrt_n + 1, 100)):
                    c_sq = n_val - e**2 - a**2 - b**2
                    if c_sq >= 0:
                        c = int(c_sq ** 0.5)
                        if c**2 == c_sq and c <= 100:
                            decompositions.append([e, a, b, c])
        
        # Teste jede Zerlegung mit allen Permutationen und Vorzeichenkombinationen
        for decomp in decompositions:
            # Überspringe, wenn es die erste Zerlegung ist (bereits getestet)
            if decomp == first_decomp:
                continue
                
            for signs in product([-1, 1], repeat=4):
                signed_decomp = [signs[i] * decomp[i] for i in range(4)]
                for perm in permutations(signed_decomp):
                    if all_coordinates_different_families_optimized(*perm):
                        return list(perm)
        
        # Keine gültige Zerlegung gefunden
        return None
        
    except Exception as e:
        return None

# ============================================================================
# HAUPTPROGRAMM - Berechnet Vierlinge neu und filtert nach vier verschiedenen Familien
# ============================================================================
filtered_results = []
total_count = 0
squarefree_count = 0
skipped_count = 0

print(f"Berechne Vierlinge für Zahlen bis {limit:,}...")
print("Nur quadratfreie Zahlen, deren Primfaktoren zu vier verschiedenen Familien (mod 12) gehören, werden behalten.\n")

for n in range(1, limit + 1):
    total_count += 1
    
    # Prüfe, ob n quadratfrei ist
    if not is_squarefree(n):
        skipped_count += 1
        continue
    
    squarefree_count += 1
    
    # Prüfe, ob die Primfaktoren zu vier verschiedenen Familien gehören
    if not prime_factors_have_four_different_families(n):
        skipped_count += 1
        continue
    
    # Zerlege n in 4 Quadrate (normale Zerlegung, keine Familienprüfung der Koordinaten)
    decomposition = four_number_sum(n)
    if not decomposition:
        skipped_count += 1
        continue
    
    e, a, b, c = decomposition
    
    # Die Zahl hat Primfaktoren mit vier verschiedenen Familien
    filtered_results.append([n, e, a, b, c])
    
    # Fortschrittsanzeige
    if total_count % progress_interval == 0:
        print(f"  {total_count:,} Zahlen geprüft, {squarefree_count:,} quadratfrei, "
              f"{len(filtered_results):,} Zahlen mit vier verschiedenen Primfaktor-Familien gefunden")

print(f"\n✓ Berechnung abgeschlossen!")
print(f"  - {total_count:,} Zahlen insgesamt geprüft")
print(f"  - {squarefree_count:,} quadratfreie Zahlen gefunden")
print(f"  - {len(filtered_results):,} Zahlen mit vier verschiedenen Primfaktor-Familien gefunden")
if squarefree_count > 0:
    print(f"  - Erfolgsrate: {float(len(filtered_results)/squarefree_count*100):.2f}% (von quadratfreien Zahlen)")
else:
    print(f"  - Erfolgsrate: 0.00%")

# Optional: Speichere Ergebnisse in CSV
if save_to_csv and len(filtered_results) > 0:
    print(f"\nSpeichere gefilterte Ergebnisse in {output_filename}...")
    with open(output_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['n', 'e', 'a', 'b', 'c'])
        writer.writerows(filtered_results)
    print(f"✓ Datei '{output_filename}' erfolgreich erstellt!")

# Zeige alle gefilterten Ergebnisse
print(f"\nAlle Vierlinge mit vier verschiedenen Primfaktor-Familien:")
print(f"{'n':<10} {'e':<8} {'a':<8} {'b':<8} {'c':<8} {'Primzahlen (nach Familien)':<30}")
print("-" * 90)
for n, e, a, b, c in filtered_results:
    # Zeige die tatsächlichen Primzahlen, gruppiert nach Familien
    from sage.all import factor
    n_int = Integer(n)
    factors = factor(n_int)
    
    # Gruppiere Primzahlen nach ihren Familien
    primes_by_family = {1: [], 5: [], 7: [], 11: []}  # e, a, b, c
    for prime, exp in factors:
        prime_val = int(prime)
        fam = get_family_mod12_fast(prime_val)
        if fam is not None:
            primes_by_family[fam].append(prime_val)
    
    # Sortiere Primzahlen innerhalb jeder Familie
    for fam in primes_by_family:
        primes_by_family[fam].sort()
    
    # Erstelle String: Primzahlen nach Familien sortiert (e, a, b, c)
    prime_list = []
    for fam in [1, 5, 7, 11]:  # e, a, b, c
        prime_list.extend(primes_by_family[fam])
    
    primes_str = ",".join(str(p) for p in prime_list)
    print(f"{n:<10} {e:<8} {a:<8} {b:<8} {c:<8} {primes_str:<30}")

In [ ]:
# ============================================================================
# CSV-EINLESEN UND PRODUKT-BERECHNUNG MIT BRIEM-ZAHLEN (PRIMFAKTOREN)
# ============================================================================
# Liest die CSV-Datei ein, faktorisiert jede Zahl n,
# gruppiert die Primfaktoren nach Familien (mod 12),
# nimmt aus jeder Familie eine Primzahl und berechnet
# das Produkt der anderen drei Familien
# ============================================================================

import csv
import os
from sage.all import factor, Integer

def get_family_mod12_fast(x):
    """
    Bestimmt die Familie einer Zahl basierend auf modulo 12.
    Gibt direkt den Rest zurück (1, 5, 7, 11) oder None.
    
    Args:
        x: Zahl (int)
    
    Returns:
        1 für x ≡ 1 (mod 12) -> Familie 'e'
        5 für x ≡ 5 (mod 12) -> Familie 'a'
        7 für x ≡ 7 (mod 12) -> Familie 'b'
        11 für x ≡ 11 (mod 12) -> Familie 'c'
        None für andere Reste
    """
    r = abs(x) % 12
    if r in (1, 5, 7, 11):
        return r
    return None

# CSV-Dateiname (kann angepasst werden)
csv_filename = "energiedoku_viertupel_gefiltert.csv"

# Prüfe, ob die Datei existiert
if not os.path.exists(csv_filename):
    print(f"⚠️  Warnung: Datei '{csv_filename}' nicht gefunden.")
    print("Bitte stellen Sie sicher, dass die CSV-Datei existiert oder den Dateinamen anpassen.")
else:
    results_with_products = []
    
    # Lese die CSV-Datei ein
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)  # Überspringe Header
        
        print(f"Lese Daten aus '{csv_filename}'...")
        print(f"Header: {header}\n")
        
        # Verarbeite jede Zeile
        for row_num, row in enumerate(reader, start=2):
            if len(row) < 5:
                print(f"⚠️  Zeile {row_num}: Unvollständige Daten, überspringe...")
                continue
            
            try:
                # Extrahiere die Werte (n, e, a, b, c)
                n = int(row[0])
                
                # Faktorisiere n, um die Primfaktoren (Briem-Zahlen) zu bekommen
                n_int = Integer(n)
                factors = factor(n_int)
                
                # Gruppiere Primzahlen nach ihren Familien
                primes_by_family = {1: [], 5: [], 7: [], 11: []}  # e, a, b, c
                for prime, exp in factors:
                    prime_val = int(prime)
                    fam = get_family_mod12_fast(prime_val)
                    if fam is not None:
                        primes_by_family[fam].append(prime_val)
                
                # Sortiere Primzahlen innerhalb jeder Familie (kleinste zuerst)
                for fam in primes_by_family:
                    primes_by_family[fam].sort()
                
                # Prüfe, ob alle vier Familien vorhanden sind
                if not all(len(primes_by_family[fam]) > 0 for fam in [1, 5, 7, 11]):
                    print(f"⚠️  Zeile {row_num}: Nicht alle vier Familien vorhanden, überspringe...")
                    continue
                
                # Nimm aus jeder Familie die erste (kleinste) Primzahl
                prime_e = primes_by_family[1][0]   # Familie e (mod 12 = 1)
                prime_a = primes_by_family[5][0]   # Familie a (mod 12 = 5)
                prime_b = primes_by_family[7][0]   # Familie b (mod 12 = 7)
                prime_c = primes_by_family[11][0]  # Familie c (mod 12 = 11)
                
                # Berechne das Produkt der anderen drei Familien für jede Familie
                # Für Familie e: Produkt von a, b, c
                prod_e = prime_a * prime_b * prime_c
                
                # Für Familie a: Produkt von e, b, c
                prod_a = prime_e * prime_b * prime_c
                
                # Für Familie b: Produkt von e, a, c
                prod_b = prime_e * prime_a * prime_c
                
                # Für Familie c: Produkt von e, a, b
                prod_c = prime_e * prime_a * prime_b
                
                # Speichere das Ergebnis
                results_with_products.append({
                    'n': n,
                    'prime_e': prime_e, 'prod_ohne_e': prod_e,
                    'prime_a': prime_a, 'prod_ohne_a': prod_a,
                    'prime_b': prime_b, 'prod_ohne_b': prod_b,
                    'prime_c': prime_c, 'prod_ohne_c': prod_c
                })
                
            except ValueError as e:
                print(f"⚠️  Zeile {row_num}: Fehler beim Konvertieren der Werte: {e}")
                continue
            except Exception as e:
                print(f"⚠️  Zeile {row_num}: Fehler: {e}")
                continue
    
    # Zeige die Ergebnisse
    if len(results_with_products) > 0:
        print(f"✓ {len(results_with_products)} Zeilen erfolgreich eingelesen und verarbeitet.\n")
        print("=" * 120)
        print("BRIEM-ZAHLEN (PRIMFAKTOREN) MIT PRODUKTEN DER ANDEREN DREI FAMILIEN")
        print("=" * 120)
        print(f"{'n':<10} {'Prime_e':<10} {'Prod(a,b,c)':<15} {'Prime_a':<10} {'Prod(e,b,c)':<15} {'Prime_b':<10} {'Prod(e,a,c)':<15} {'Prime_c':<10} {'Prod(e,a,b)':<15}")
        print("-" * 120)
        
        for result in results_with_products:
            print(f"{result['n']:<10} "
                  f"{result['prime_e']:<10} {result['prod_ohne_e']:<15} "
                  f"{result['prime_a']:<10} {result['prod_ohne_a']:<15} "
                  f"{result['prime_b']:<10} {result['prod_ohne_b']:<15} "
                  f"{result['prime_c']:<10} {result['prod_ohne_c']:<15}")
        
        print("\n" + "=" * 120)
        print("Erklärung:")
        print("  - Prime_e, Prime_a, Prime_b, Prime_c: Primzahlen (Briem-Zahlen) aus den jeweiligen Familien (mod 12)")
        print("  - Prod(a,b,c): Produkt der Primzahlen aus den anderen drei Familien für Familie e")
        print("  - Prod(e,b,c): Produkt der Primzahlen aus den anderen drei Familien für Familie a")
        print("  - Prod(e,a,c): Produkt der Primzahlen aus den anderen drei Familien für Familie b")
        print("  - Prod(e,a,b): Produkt der Primzahlen aus den anderen drei Familien für Familie c")
        print("=" * 120)
    else:
        print("⚠️  Keine Daten gefunden in der CSV-Datei.")

In [11]:
import numpy as np

# --- Basisparameter ---
pi = np.pi
lead = 6 * pi**5

# Diskretisierungsparameter
N = 50000  # spektrale Auflösung (variieren!)

# --- Operator-Bausteine (Modelliert, nicht gefittet!) ---

def D0(N):
    # freier Dirac-artiger Operator (toy model)
    return np.diag(np.linspace(1, N, N))

def V_geom(N):
    # geometrische Volumenkorrektur ~ 1/N
    return np.diag(np.ones(N) / N)

def V_fam(N, strength=1e-2):
    # schwache Familienkopplung
    return strength * np.random.randn(N, N) / N

def V_iso(N, strength=1e-3):
    # Isospin/Neutralitätskopplung
    return strength * np.random.randn(N, N) / N


In [ ]:
# Proton
D_p = D0(N) + V_geom(N) + V_fam(N)
eig_p = np.linalg.eigvalsh(D_p)

# Neutron
D_n = D0(N) + V_geom(N) + V_fam(N) + V_iso(N)
eig_n = np.linalg.eigvalsh(D_n)

# Elektron: lokaler Grundmodus
lambda_e = eig_p[0]

# Proton/Neutron-Moden (erste stabile kollektive)
lambda_p = eig_p[10]   # Beispielindex
lambda_n = eig_n[10]


In [13]:
# zwei rein geometrische Moden
mu1 = eig_p[20]
mu2 = eig_p[40]

R = mu2 / mu1
print("Interner Quotient:", R)


Interner Quotient: 1.9523800120969703


In [ ]:
eps_p = lambda_p / (lead * lambda_e) - 1
eps_n = lambda_n / (lead * lambda_e) - 1

print("ε_p =", eps_p)
print("ε_n =", eps_n)


ε_p = -0.9940092092450461
ε_n = -0.9940092092621206
